# Phase 2B: Final Mountain Multi-View Teacher

This notebook builds on completed Phase 2A outputs. It does **not** re-extract AST/Whisper unless those files are missing. It tries the highest-value Phase 2B upgrades: optional AudioJEPA-lite features, tuned RBF SVMs, validation-weighted probability ensemble, and rare-class specialists.

## 0. Mount Drive and Locate Project

In [ ]:
import os
from pathlib import Path

try:
    from google.colab import drive
    IN_COLAB = True
except Exception:
    IN_COLAB = False

if IN_COLAB:
    drive.mount('/content/drive')

PROJECT_DIR = Path('/content/drive/MyDrive/Infant-State-Recognition-System') if IN_COLAB else Path.cwd().parent
assert (PROJECT_DIR / 'src' / 'phase2b').exists(), f'Missing Phase 2B code: {PROJECT_DIR}'
os.chdir(PROJECT_DIR)
print('Working directory:', Path.cwd())

## 1. Install Dependencies and Verify Phase 2A Feature Banks

In [ ]:
!pip install -q librosa==0.10.1 soundfile==0.12.1 transformers accelerate scikit-learn pandas scipy tqdm joblib matplotlib seaborn

import sys
import pandas as pd
from pathlib import Path

if str(PROJECT_DIR) not in sys.path:
    sys.path.insert(0, str(PROJECT_DIR))

feature_dir = PROJECT_DIR / 'data_lake' / 'features' / 'phase2a'
required = [
    'ast_embeddings.npz',
    'ast_aux_adapted_embeddings.npz',
    'whisper_embeddings.npz',
    'handcrafted_features.npz',
    'ablation_no_aux_no_whisper_features.npz',
    'ablation_aux_no_whisper_features.npz',
    'ablation_no_aux_with_whisper_features.npz',
    'ablation_aux_with_whisper_features.npz',
]
missing = [name for name in required if not (feature_dir / name).exists()]
print('Feature dir:', feature_dir)
print('Missing required Phase 2A feature banks:', missing)
assert not missing, 'Run Phase2A_Pretrained_Feature_Bank.ipynb first, then rerun Phase 2B.'

## 2. Optional AudioJEPA Feature Branch

This is the only new GPU-heavy branch. It trains a JEPA-style latent prediction model on approved train/auxiliary audio: context spectrogram blocks go through an online encoder, clean target blocks go through an EMA target encoder, and a predictor learns target patch embeddings in latent space. It exports `jepa_lite_embeddings.npz`. If time is tight, set `RUN_JEPA_LITE = False`; the rest of Phase 2B still runs.

In [ ]:
RUN_JEPA_LITE = True
JEPA_EPOCHS = 8
JEPA_BATCH_SIZE = 32

jepa_path = feature_dir / 'jepa_lite_embeddings.npz'
if RUN_JEPA_LITE and not jepa_path.exists():
    !python scripts/phase2b_jepa_lite.py --epochs {JEPA_EPOCHS} --batch-size {JEPA_BATCH_SIZE}
elif jepa_path.exists():
    print('Found existing JEPA-lite embeddings:', jepa_path)
else:
    print('Skipping JEPA-lite branch.')

## 2b. Test-Time Augmentation Feature Banks

Re-extract AST and (if present) auxiliary-adapted AST embeddings from multiple deterministic temporal crops of each canonical 10-second clip and average them in embedding space. This is a low-risk way to squeeze a small but real macro-F1 gain out of fixed encoders without retraining anything. The new banks (`ast_tta_embeddings.npz`, `ast_aux_tta_embeddings.npz`) are picked up automatically by Phase 2B SVM tuning.

In [ ]:
RUN_TTA = True

if RUN_TTA:
    !python scripts/phase2b_tta_features.py --crop-seconds 7.0 --num-crops 4 --batch-size 8
else:
    print('Skipping TTA feature banks.')

## 3. Tune RBF SVMs Across Feature Views

In [ ]:
# Full grid with repeated train-only CV. If Colab time is tight, use e.g. --max-configs 48.
!python scripts/phase2b_tune_svm.py --top-k 3 --max-configs 0 --cv-folds 3 --cv-repeats 2

## 4. Build Validation-Weighted Probability Ensemble

In [ ]:
!python scripts/phase2b_weighted_ensemble.py --top-n 8 --random-candidates 5000 --seed 42

## 5. Rare-Class Specialist Override

In [ ]:
!python scripts/phase2b_rare_specialists.py

## 6. Repeated-Split Phase 2B Teacher Evaluation

The single fixed-split number from sections 4 and 5 is sensitive to which 17 belly-pain / 13 burping samples land where. To compare honestly with Phase 2A's repeated-split number (`mean=0.6566 ± 0.048`), we re-stratify train/val/test across multiple seeds, refit the previously-tuned top configs, rebuild the weighted ensemble, retrain rare-class specialists, and report `mean ± std` of the final test macro-F1.

In [ ]:
!python scripts/phase2b_repeated_evaluation.py --n-seeds 5 --top-k-per-bank 2 --top-n-ensemble 8 --random-candidates 2000

## 7. Final Deployment Report

Bootstrap 95% confidence intervals on macro-F1 / balanced accuracy / per-class F1, source-stratified macro-F1 (per `source_dataset`), top-1 calibration ECE, and the abstention curve (macro-F1 vs coverage as we drop low-confidence predictions). This is the honest "what would a clinical / product reviewer see" view.

In [ ]:
!python scripts/phase2b_final_report.py --n-bootstrap 2000

## 8. All Metrics Inspection

In [ ]:
import json
from pathlib import Path

metrics_dir = PROJECT_DIR / 'results' / 'phase2b' / 'metrics'
for path in sorted(metrics_dir.glob('*.json')):
    print('\n' + '=' * 90)
    print(path.name)
    data = json.loads(path.read_text())
    if 'weighted_ensemble' in path.name or 'rare_specialist' in path.name:
        print('validation macro-F1:', data['validation']['macro_f1'])
        print('test macro-F1:', data['test']['macro_f1'])
        print('test per-class F1:', {k: round(v['f1'], 4) for k, v in data['test']['per_class'].items()})
    elif 'svm_tuning_summary' in path.name:
        best = []
        for feature_name, result in data.items():
            for model_name, row in result.get('final_models', {}).items():
                best.append((row['test']['macro_f1'], feature_name, model_name))
        for score, feature_name, model_name in sorted(best, reverse=True)[:10]:
            print(f'{score:.4f} | {feature_name} | {model_name}')